## Solving underdetermined linear equations

Example: solving $\mathbf A \vec x = \vec b$, where $\mathbf A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$, $\vec b = \begin{bmatrix} 7 \\ 8 \end{bmatrix}$.

From linear algebra, the general solution should be $\displaystyle\vec x = \frac{1}{3}\begin{bmatrix} -19 \\ 20 \\ 0 \end{bmatrix} + t\begin{bmatrix} 1 \\ -2 \\ 1 \end{bmatrix} $.

Obviously, the solution with minimal $\Vert \vec x\Vert^2$ is fund when $\displaystyle t=\frac{59}{18}$, and $\displaystyle x_\text{min norm} = \frac{1}{18} \begin{bmatrix} -55 \\ 2 \\ 59 \end{bmatrix} \approx \begin{bmatrix} -3.056 \\ 0.1111 \\ 3.278 \end{bmatrix}$, or the least square solution.

In [1]:
# define the linear equations
import numpy as np

A = np.array([[1, 2, 3], 
              [4, 5, 6]])
b = np.array([7, 8])

In [2]:
# calculate the least square solution
x_lstsq, resids, rank, s = np.linalg.lstsq(A, b, rcond=None)
print("the least square solution is")
print(x_lstsq)

the least square solution is
[-3.05555556  0.11111111  3.27777778]


To solve $\mathbf A \vec x = \vec b$, where $\mathbf A$ is an $m \times n$ matrix, using $m$ equations to solve $n$ unknowns. If $\mathbf A^T \mathbf A$ is invertible, the least square solution can be easily calculated as $\vec x_\text{lstsq} = (\mathbf A^T \mathbf A)^{-1} \mathbf A^T \vec b$.

Here, unfortunately, $\mathbf A$ is $2 \times 3$, and $\mathbf A^T \mathbf A$ is $3 \times 3$ but $\text{rank}(\mathbf A^T \mathbf A)=2$ (not invertible). However, we can approximate the least square solution by adding a small regularization $\epsilon > 0$ and find the least square solution of $\Vert \mathbf A \vec x - \vec b\Vert^2 + \epsilon \Vert \vec x \Vert^2$. Now we should have $\vec x_\text{lstsq} = (\mathbf A^T \mathbf A + \epsilon \mathbf I)^{-1} \mathbf A^T \vec b$ and $\mathbf A^T \mathbf A + \epsilon \mathbf I$ is invertible.

If we choose $\epsilon$ as a VERY small positive value, the solution should be the least square solution!

In [3]:
# reg param as small as 1e-6
epsilon = 0.000001

# calculate (A^T A + epsilon I) and A^T b
A_T_A = np.dot(A.T, A)
A_T_b = np.dot(A.T, b)
epsilon_I = epsilon * np.eye(A.shape[1])

x_reg = np.linalg.inv(A_T_A + epsilon_I).dot(A_T_b)

print("the least square solution from solving with regularization：")
print(x_reg)

the least square solution from solving with regularization：
[-3.05554968  0.11111192  3.27777353]


Why does the small $\epsilon$ work? Let's look at $\mathbf A$'s SVD. $\mathbf A = \mathbf U \mathbf \Sigma \mathbf V^T$. If $\mathbf A^T \mathbf A$ is invertible, then $\mathbf \Sigma$ is invertible, then $\vec x = \mathbf V \mathbf \Sigma^{-1} \mathbf U^T \vec b = \mathbf V \text{diag}(\frac{1}{\sigma_i}) \mathbf U^T \vec b$. Unfortunately, here is not. But, $\vec x_\text{lstsq} = \mathbf V (\mathbf \Sigma^T \mathbf \Sigma + \epsilon \mathbf I )^{-1} \mathbf \Sigma^T \mathbf U^T \vec b = \mathbf V \text{diag}(\frac{\sigma_i}{\sigma_i^2 + \epsilon}) \mathbf U^T \vec b$. When $\epsilon \to 0$, $\vec x_\text{lstsq} \to$ the solution that satisfies $\mathbf A \vec x = \vec b$.

In [5]:
U, S, Vt = np.linalg.svd(A)

print("U =\n", U)
print("Singular values =", S)
print("V^T =\n", Vt)

Sigma = np.zeros_like(A, dtype=float)
Sigma[:len(S), :len(S)] = np.diag(S)

U =
 [[-0.3863177  -0.92236578]
 [-0.92236578  0.3863177 ]]
Singular values = [9.508032   0.77286964]
V^T =
 [[-0.42866713 -0.56630692 -0.7039467 ]
 [ 0.80596391  0.11238241 -0.58119908]
 [ 0.40824829 -0.81649658  0.40824829]]


Denote $\mathbf P = (\mathbf \Sigma^T \mathbf \Sigma + \epsilon \mathbf I )^{-1} \mathbf \Sigma^T$, then $\vec x_\text{lstsq} = \mathbf V \mathbf P \mathbf U^T \vec b$.

In [7]:
def P(Sigma, epsilon):
    return np.linalg.inv(np.dot(Sigma.T, Sigma) + epsilon * np.eye(A.shape[1])).dot(Sigma.T)

In [ ]:
epsilon_large = 1e-3
x_epsilon_large = Vt.T @ P(Sigma, epsilon_large) @ U @ b
print(x_epsilon_large)

epsilon_medium = 1e-6
x_epsilon_medium = Vt.T @ P(Sigma, epsilon_medium) @ U @ b
print(x_epsilon_medium)

epsilon_small = 1e-9
x_epsilon_small = Vt.T @ P(Sigma, epsilon_small) @ U @ b
print(x_epsilon_small)

[-3.04969398  0.1119225   3.27353898]
[-3.05554968  0.11111192  3.27777353]
[-3.05555555  0.11111111  3.27777777]
